In [1]:
# import the required packages
import pandas as pd
import numpy as np

In [2]:
# Load the SQL toolset extension
%load_ext sql

In [3]:
#2. Connect to (and create) the database file
%sql sqlite:///bike_orders_database.sqlite

Connecting to 'sqlite:///bike_orders_database.sqlite'

In [6]:
def collect_data():
    """
    Collects and combines the bike orders data using the jupyter-sql magic extension.
    """
    
    # 1.0 Connect to database and load tables via SQL magic
    # Using the %sql magic command to fetch data directly into DataFrames
    bikes     = %sql SELECT * FROM bikes
    bikeshops = %sql SELECT * FROM bikeshops
    orderlines = %sql SELECT * FROM orderlines
    
    # Convert to DataFrames and drop 'index' column
    data_dict = {
        'bikes': bikes.DataFrame().drop("index", axis=1),
        'bikeshops': bikeshops.DataFrame().drop("index", axis=1),
        'orderlines': orderlines.DataFrame().drop("index", axis=1)
    }

    # 2.0 Combining Data
    joined_df = data_dict['orderlines'] \
        .merge(
            right    = data_dict['bikes'],
            how      = 'left',
            left_on  = 'product.id',
            right_on = 'bike.id'
        ) \
        .merge(
            right    = data_dict['bikeshops'],
            how      = "left",
            left_on  = "customer.id",
            right_on = 'bikeshop.id'
        )

    # 3.0 Cleaning Data 
    df = joined_df
    df['order.date'] = pd.to_datetime(df['order.date'])

    temp_df = df['description'].str.split(" - ", expand = True)
    df['category.1'] = temp_df[0]
    df['category.2'] = temp_df[1]
    df['frame.material'] = temp_df[2]

    temp_df = df['location'].str.split(", ", expand = True)
    df['city'] = temp_df[0]
    df['state'] = temp_df[1]

    df['total.price'] = df['quantity'] * df['price']

    cols_to_keep_list = [
        'order.id', 'order.line', 'order.date',    
        'quantity', 'price', 'total.price', 
        'model', 'category.1', 'category.2', 'frame.material', 
        'bikeshop.name', 'city', 'state'
    ]

    df = df[cols_to_keep_list]
    df.columns = df.columns.str.replace(".", "_")

    return df

In [7]:
df = collect_data()

Running query in 'sqlite:///bike_orders_database.sqlite'

Running query in 'sqlite:///bike_orders_database.sqlite'

Running query in 'sqlite:///bike_orders_database.sqlite'

In [8]:
df

,order_id,order_line,order_date,quantity,price,total_price,model,category_1,category_2,frame_material,bikeshop_name,city,state
0,1,1,2011-01-07,1,6070,6070,Jekyll Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY
1,1,2,2011-01-07,1,5970,5970,Trigger Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY
2,2,1,2011-01-10,1,2770,2770,Beast of the East 1,Mountain,Trail,Aluminum,Kansas City 29ers,Kansas City,KS
3,2,2,2011-01-10,1,5970,5970,Trigger Carbon 2,Mountain,Over Mountain,Carbon,Kansas City 29ers,Kansas City,KS
4,3,1,2011-01-10,1,10660,10660,Supersix Evo Hi-Mod Team,Road,Elite Road,Carbon,Louisville Race Equipment,Louisville,KY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15639,2000,4,2015-12-25,1,2660,2660,CAAD Disc Ultegra,Road,Elite Road,Aluminum,Austin Cruisers,Austin,TX
15640,2000,5,2015-12-25,1,1350,1350,Trail 2,Mountain,Sport,Aluminum,Austin Cruisers,Austin,TX
15641,2000,6,2015-12-25,1,1680,1680,CAAD12 105,Road,Elite Road,Aluminum,Austin Cruisers,Austin,TX
15642,2000,7,2015-12-25,1,2880,2880,F-Si Carbon 4,Mountain,Cross Country Race,Carbon,Austin Cruisers,Austin,TX


# 1. How Python works - Objects

In [ ]:
# Get the object's class
type(df)

In [ ]:
# Objects have classes
# type.mro() gets the object's inheritance structure in the order that methods are searched for.
# In Python, Objects get methods and attributes from the classes that they inherit from.
# Low Level: Python bultin Pbject classes like Int, Float, String, List, Tuple are lower level classes.
# High Level> Pandas Data Frames and Series are high-level objects. They have much more depth to their inheritance
# and therefore have expanded functionality
type(df).mro() # high level
type("string").mro() # low level

In [ ]:
# Objects have atrributes
# Attributes: the metadata that exists for an object. Think of the dimensions of the data frama has an attribute
# called shape. We don't put parenthesis behind attributes.
df.shape
df.columns

In [ ]:
# Objects have methods
# We put parentheses around methods
df.query("model == 'Jekyll Carbon 2'")

# 2. Key Data Structures for Analysis

In [ ]:
# Pandas Data Frame.- The key structure that holds column of Pandas Series.
# The DF has column and index attributes.
# The DF has many methods that enable data analysis
df
type(df)

In [ ]:
# Pandas Series
# Passing a single text string representing a column name returns a Pandas Series
# Many of the method names are the same between Pandas Data Frames and Series, but these are actually different
# methods and often have different arguments.
df['order_date']
type(df['order_date'])
df['order_date'].dt.year
df.dt # We will get an error because the dataframe has no attribute 'dt'

In [ ]:
# Numpy Arrays
# Pandas Series are built on top of Numpy Arrays.  The Series adds an index and metadata like series name.
# The Numpy Array then provides the core functionality, like np.sum()

# Pandas and Numpy work together. 
# Pandas adds the data frame and series structure along with methods to work with data.
# Numpy adds the low-level functionality to perform the work like: sum, mean, std, log, etc.
df['order_date'].values
type(df['order_date'].values)
type(df['order_date'].values).mro()


In [ ]:
# Data Types
# Numpy data types (extend Python builtin data types). 
# Numpyuses special data types (e.g. int64) that extend the Python builtin data types (e.g. int)
# Generally, the Numpy versions are optimized for memory allocation, which is why you see 'int64', an integer that can take
# on 64-bits of memory: integer(-9223372036854775808 to 9223372036854775807) 
df['price'].values.dtype

# 3. Data Structures - Python
Pyhton "Built-in" libraries contain many data structures, data types, and functions that are automatically loaded and always available.  These come from the "builtin" module

In [ ]:
# Dictionaries
# Contains a key-value pair. It is used commonly in pandas for assignment.
# Dictionaries are used in the df.rename() and df.agg() functions for column renaming and aggregations
d = {'a':1}
type(d)
d.keys()
d.values()

In [ ]:
# Select items from Dictionaries
# This is very similar to how we select a Pandas Series by column from a Data Frame
d['a']

In [ ]:
# Lists
# We commonly access elements in lists by their position (index location)
[1, "A", [2, "B"]]
l = [1, "A", [2, "B"]]
l[0]

# List conversion (casting). It is very common to turn non-iterable object into iterable objects.
d.values() # It returns a list. To gain access to that list, one way to do is to wrap around a list
list(d.values())
list(d.values())[0]

In [ ]:
# Tuples. An immutable list
# Tuples are used in pandas for storing data frame shape and multi-index column names.
df.shape
type(df.shape)
type(df.shape).mro()

# Tuples are immutable.  We can't change elements of a tuple.  We need to overwrite the entire tuple
t = (10,20)
type(t)
t[0]
t[0] = 20 # we will get an error

In [ ]:
# Base Data Types
# The main data types we will work with are: str = string, int = integer, float = numeric (floating-point) decimal,
# bool = True/False
1.5
type(1.5)
type(1.5).mro()

1
type(1)

df.total_price # Numpy is built for performance. It has a special "int64" class that helps improved speed and memory efficiency
df.total_price.dtype
df.total_price.values

# The "object" data type commonly refers to string or categorical data
df['model'].values
df['model'].values[0]  # Grab the first value
type(df['model'].values[0])

In [ ]:
# Casting
# Converting data from one type to another.  Example: Datetime to String.
model = "Jekyll Carbon 2"
price = 6070
f"The first model is: {model}"
f"The price of the first model: {price}"
# price + "some text"  # I will get an error
str(price) + " some text"
"50"
int(50)

In [ ]:
# Casting sequences
# Go from low-level to high-level objects
range(1,50)
type(range(1,50))
type(range(1,50)).mro() # Range is a generator. If we want to grab the seq, we need to cast it to a list
list(range(1,50))
r = list(range(1,50))
np.array(r)
pd.Series(r)  # Cast to a Series
pd.Series(r).to_frame()

In [ ]:
# Data Frame column DType Conversions
# The principle of data cleaning is derived from your ability to convert columns of data to the appropriate
# data type (dtype)  for your analysis
# This involves dtype conversion
# s.astype() converts a series from one dtype to another dtype
df['order_date'].astype('str')
df['order_date'].astype('str').str.replace("-", "/")

In [ ]:
# Close the connection
%sql --close sqlite:///bike_orders_database.sqlite